# Version 3 — train-only EEG preprocessing ablations

This version tests whether input scale causes the seed-sensitive collapse observed in Version 2. It compares four predeclared methods—raw, `log1p`, per-channel z-score, and robust clipping/scaling—without changing the participants, seeds, CNN, validation tail, learning rate, or early stopping.

All fitted statistics use the fit partition only. Validation and held-out-participant data are transformed with those frozen statistics. Padding and missing-fixation zeros stay zero.

## 1. Setup and GPU preflight

Select **Runtime → Change runtime type → GPU** before running. This notebook stops instead of silently training on CPU.

In [ ]:
import os, sys, subprocess

REPO_SLUG = 'parmisbathayan/reproduce-teco-results'
REPO_URL = f'https://github.com/{REPO_SLUG}.git'
REPO_DIR = 'reproduce-teco-results'

if not os.path.isdir('src'):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
sys.path.insert(0, 'src')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('No GPU found. Select Runtime > Change runtime type > GPU, reconnect, and rerun.')

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('working directory:', os.getcwd())
print('Git commit:', commit)
print('TensorFlow:', tf.__version__, '| GPUs:', [gpu.name for gpu in gpus])

## 2. Locked configuration and Drive paths

The full run is 4 preprocessing methods × 5 seeds × 12 participants = **240 folds**. It is crash-safe and resumes from `fold_metrics.csv`. Do not change the configuration after inspecting results.

In [ ]:
import json

trt_dir = '/content/drive/MyDrive/Thesis/Data/TRT_Total'
labels_csv = '/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv'
out_dir = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly/v3_preprocessing'
v2_metrics_csv = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly/v2_diagnoseCollapse/fold_metrics.csv'
config_path = 'configs/v3_preprocessing.json'

with open(config_path) as stream:
    config = json.load(stream)
expected_folds = (len(config['preprocessing_methods']) * len(config['seeds']) * 12)
assert expected_folds == 240
config

## 3. Run or resume Version 3

The raw control runs first. Each completed method/seed/participant fold is saved immediately. If Colab disconnects, rerun the setup, paths, and experiment cells; completed folds will be skipped.

In [ ]:
from v3_preprocessing import run_v3_preprocessing

metrics = run_v3_preprocessing(trt_dir, labels_csv, out_dir, config_path)

## 4. Completion and primary comparison

In [ ]:
import pandas as pd
from IPython.display import display

fold_metrics = pd.read_csv(os.path.join(out_dir, 'fold_metrics.csv'))
by_method = pd.read_csv(os.path.join(out_dir, 'summary_by_method.csv'))
by_method_seed = pd.read_csv(os.path.join(out_dir, 'summary_by_method_seed.csv'))

print('completed folds:', len(fold_metrics), '/', expected_folds)
display(by_method.sort_values(['collapse_rate', 'f1_mean'], ascending=[True, False]).round(4))
display(by_method_seed.round(4))

## 5. Verify the raw control against Version 2

With the same TensorFlow environment, v3's raw rows should reproduce v2. Any differences remain visible rather than being silently discarded.

In [ ]:
v2 = pd.read_csv(v2_metrics_csv)[['seed', 'participant', 'f1', 'collapsed']]
raw = (fold_metrics.query("preprocessing == 'raw'")
       [['seed', 'participant', 'f1', 'collapsed']])
control = raw.merge(v2, on=['seed', 'participant'], suffixes=('_v3_raw', '_v2'))
control['f1_difference'] = control.f1_v3_raw - control.f1_v2
print('matched raw folds:', len(control), '/ 60')
print('maximum absolute F1 difference:', control.f1_difference.abs().max())
print('collapse disagreements:', (control.collapsed_v3_raw != control.collapsed_v2).sum())
display(control.loc[control.f1_difference.abs() > 1e-9].round(6))

## 6. Paired improvements over raw

In [ ]:
paired = pd.read_csv(os.path.join(out_dir, 'paired_f1_comparison.csv'))
delta_columns = [column for column in paired if column.endswith('_minus_raw')]
delta_summary = pd.DataFrame({
    'mean_delta': paired[delta_columns].mean(),
    'median_delta': paired[delta_columns].median(),
    'improved_folds': (paired[delta_columns] > 0).sum(),
    'worsened_folds': (paired[delta_columns] < 0).sum(),
})
display(delta_summary.round(4))

## 7. Presentation-ready figures

In [ ]:
import matplotlib.pyplot as plt

order = config['preprocessing_methods']
plot_data = by_method.set_index('preprocessing').loc[order].reset_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(plot_data.preprocessing, plot_data.f1_mean,
            yerr=plot_data.f1_std, capsize=3)
axes[0].axhline(0.6666, color='tab:red', linestyle='--', label='Thesis mean')
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Macro-F1 across 60 folds')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend()
axes[1].bar(plot_data.preprocessing, plot_data.collapse_rate, color='tab:orange')
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('One-class collapse rate')
axes[1].tick_params(axis='x', rotation=20)
fig.suptitle('Version 3: train-only preprocessing ablations')
fig.tight_layout()
figures_dir = os.path.join(out_dir, 'figures')
os.makedirs(figures_dir, exist_ok=True)
fig.savefig(os.path.join(figures_dir, 'v3_preprocessing_summary.png'),
            dpi=200, bbox_inches='tight')
plt.show()

## Output inventory

- `fold_metrics.csv`: one row per preprocessing/seed/participant fold (240 when complete).
- `summary_by_method.csv`, `summary_by_method_seed.csv`, `summary_by_method_participant.csv`: primary aggregate comparisons.
- `paired_f1_comparison.csv`: paired F1 changes relative to raw for each seed and participant.
- `predictions/` and `histories/`: sentence predictions and epoch histories for every fold.
- `preprocessing_parameters/`: train-fitted per-channel centers, scales, and clipping thresholds.
- `run_metadata.json`: environment, commit, configuration, and data/source hashes.
- `figures/v3_preprocessing_summary.png`: final comparison figure.